In [35]:
import pandas as pd
import geopandas as gpd
from difflib import SequenceMatcher

In [13]:
df = pd.read_csv(r"df_clusters.csv")

In [19]:
df.head()

,region_name,mo,year,Здоровье_share,Маркетплейсы_share,Общественное питание_share,Продовольствие_share,Транспорт_share,Средняя зарплата,population,urban_share,share_industry,share_agriculture,share_services,node,cluster
0,Алтайский край,Алейский муниципальный район,2024,0.044537,0.136786,0.015952,0.548151,0.056736,44739.9,13794.0,0.000000,0.000000,0.256097,0.532495,"('Алейский муниципальный район', 'Алтайский кр...",4
1,Алтайский край,Алтайский муниципальный район,2024,0.038585,0.155752,0.027256,0.428631,0.071270,44459.5,26284.0,0.000000,0.064646,0.066836,0.860969,"('Алтайский муниципальный район', 'Алтайский к...",0
2,Алтайский край,Баевский муниципальный район,2024,0.049679,0.183845,0.006053,0.536821,0.049062,44787.8,7821.0,0.000000,0.060850,0.000000,0.729266,"('Баевский муниципальный район', 'Алтайский кр...",3
3,Алтайский край,Бийский муниципальный район,2024,0.047182,0.149325,0.027043,0.434394,0.077104,48559.1,31583.0,0.000000,0.125820,0.316038,0.521954,"('Бийский муниципальный район', 'Алтайский край')",4
4,Алтайский край,Благовещенский муниципальный район,2024,0.049559,0.148734,0.035925,0.437150,0.053833,49998.5,25464.0,0.626257,0.316731,0.052407,0.627371,"('Благовещенский муниципальный район', 'Алтайс...",1


In [6]:
gdf = gpd.read_file("Municipalities_Russia_2025.geojson")

In [36]:
gdf.head()

,region,type,name,admcen,fo,arctic,okato,oktmo,geometry,name_for_merge
0,Алтайский край,муниципальный район,Алейский муниципальный район,г Алейск,Сибирский,0,01201000000,01601000000,"MULTIPOLYGON (((9261395.767 6873159.859, 92674...",Алейский муниципальный район
1,Алтайский край,муниципальный район,Алтайский муниципальный район,с Алтайское,Сибирский,0,01202000000,01602000000,"MULTIPOLYGON (((9551392.113 6809974.372, 95511...",Алтайский муниципальный район
2,Алтайский край,муниципальный район,Баевский муниципальный район,с Баево,Сибирский,0,01203000000,01603000000,"MULTIPOLYGON (((9025684.894 7055484.739, 90236...",Баевский муниципальный район
3,Алтайский край,муниципальный район,Бийский муниципальный район,г Бийск,Сибирский,0,01204000000,01604000000,"MULTIPOLYGON (((9582010.127 6948721.29, 958356...",Бийский муниципальный район
4,Алтайский край,муниципальный район,Благовещенский муниципальный район,рп Благовещенка,Сибирский,0,01205000000,01605000000,"MULTIPOLYGON (((8971203.02 6997278.35, 8968938...",Благовещенский муниципальный район


In [20]:
gdf_names = gdf['name'].dropna().astype(str).unique()

with open('gdf_names.txt', 'w', encoding='utf-8') as f:
    for n in sorted(gdf_names):
        f.write(str(n) + '\n')

df_names = df['mo'].dropna().astype(str).unique()

with open('df_names.txt', 'w', encoding='utf-8') as f:
    for n in sorted(df_names):
        f.write(str(n) + '\n')


In [21]:
import re


with open('gdf_names.txt', encoding='utf-8') as f:
    gdf_names = [l.strip() for l in f if l.strip()]
with open('df_names.txt', encoding='utf-8') as f:
    df_names = [l.strip() for l in f if l.strip()]

def strip_type(name):
    """Убираем тип МО, оставляем только суть"""
    s = name.lower().strip()
    s = re.sub(r'\b(муниципальный|муниципальное|городской|городское|сельское|сельский)\b', '', s)
    s = re.sub(r'\b(район|округ|поселение|город|г\.|пос\.|пгт)\b', '', s)
    s = re.sub(r'[^\w\s\-]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

gdf_by_core = {}
for g in gdf_names:
    core = strip_type(g)
    gdf_by_core.setdefault(core, []).append(g)

# Для каждого df_name ищем в gdf_names
matches = []
for d in df_names:
    core_d = strip_type(d)
    if core_d in gdf_by_core:
        for g in gdf_by_core[core_d]:
            if g != d:
                matches.append((d, g, 1.0))
    else:
        best_score, best_g = 0, None
        for g in gdf_names:
            score = SequenceMatcher(None, core_d, strip_type(g)).ratio()
            if score > best_score:
                best_score, best_g = score, g
        if best_score > 0.9:
            matches.append((d, best_g, best_score))

# Только те, где названия отличаются
result = [(d, g, s) for d, g, s in matches if d != g and s > 0.9]
result.sort(key=lambda x: -x[2])

with open('replace_map.csv', 'w', encoding='utf-8') as f:
    f.write('df_name;gdf_name;score\n')
    for d, g, s in result:
        f.write(f'{d};{g};{s:.2f}\n')

print(f"Найдено пар: {len(result)}")
print("Файл: replace_map.csv")

Найдено пар: 380
Файл: replace_map.csv


In [2]:
import pandas as pd
from difflib import SequenceMatcher
import re

Нужно изменить названия мунициальных образований в gdf на те, которые есть в df. Составляем список таких образований

In [14]:


# Читаем названия из файлов
with open('df_names.txt', encoding='utf-8') as f:
    df_list = [l.strip() for l in f if l.strip()]

with open('gdf_names.txt', encoding='utf-8') as f:
    gdf_list = [l.strip() for l in f if l.strip()]

print(f"df: {len(df_list)} названий")
print(f"gdf: {len(gdf_list)} названий")

def strip_type(name):
    s = name.lower().strip()
    s = re.sub(r'\b(муниципальный|муниципальное|городской|городское|сельское|сельский)\b', '', s)
    s = re.sub(r'\b(район|округ|поселение|город|г\.|пос\.|пгт)\b', '', s)
    s = re.sub(r'[^\w\s\-]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

rows = []
for d in df_list:
    core_d = strip_type(d)
    best_score, best_g = 0, None
    for g in gdf_list:
        score = SequenceMatcher(None, core_d, strip_type(g)).ratio()
        if score > best_score:
            best_score, best_g = score, g
    if best_score > 0.9:
        rows.append({'df_name': d, 'gdf_name': best_g, 'score': round(best_score, 2)})

pairs = pd.DataFrame(rows)
print(f"Найдено пар: {len(pairs)}")

# Чёрный список (разные МО с одинаковым названием, которые не нужно менять)
blacklist = {
    ('Волжский муниципальный район', 'город Волжский'),
    ('Гусь-Хрустальный муниципальный район', 'город Гусь-Хрустальный'),
    ('Слободской муниципальный район', 'город Слободской'),
    ('Октябрьский муниципальный район', 'город Октябрьский'),
    ('Октябрьский муниципальный округ', 'город Октябрьский'),
    ('Троицкий муниципальный район', 'Троицкий'),
    ('Троицкий муниципальный район', 'Троицкий муниципальный округ'),
    ('Ленинский муниципальный район', 'Ленинский'),
    ('Ленинский муниципальный район', 'Ленинский муниципальный округ'),
    ('Советский муниципальный район', 'Советский'),
    ('Советский муниципальный округ', 'Советский'),
    ('Советский муниципальный район', 'Советский муниципальный округ'),
    ('Советский муниципальный округ', 'Советский муниципальный район'),
    ('Кировский муниципальный район', 'Кировский'),
    ('Кировский муниципальный округ', 'Кировский'),
    ('Кировский муниципальный район', 'Кировский муниципальный округ'),
    ('Кировский муниципальный округ', 'Кировский муниципальный район'),
    ('Петровский муниципальный район', 'Петровский'),
    ('Петровский муниципальный округ', 'Петровский'),
    ('Петровский муниципальный район', 'Петровский муниципальный округ'),
    ('Петровский муниципальный округ', 'Петровский муниципальный район'),
    ('Павловский муниципальный район', 'Павловский'),
    ('Павловский муниципальный округ', 'Павловский'),
    ('Павловский муниципальный район', 'Павловский муниципальный округ'),
    ('Павловский муниципальный округ', 'Павловский муниципальный район'),
    ('Первомайский муниципальный район', 'Первомайский'),
    ('Первомайский муниципальный округ', 'Первомайский'),
    ('Первомайский муниципальный район', 'Первомайский муниципальный округ'),
    ('Первомайский муниципальный округ', 'Первомайский муниципальный район'),
    ('Знаменский муниципальный район', 'Знаменский'),
    ('Знаменский муниципальный округ', 'Знаменский'),
    ('Знаменский муниципальный район', 'Знаменский муниципальный округ'),
    ('Знаменский муниципальный округ', 'Знаменский муниципальный район'),
    ('Никольский муниципальный район', 'Никольский'),
    ('Никольский муниципальный округ', 'Никольский'),
    ('Никольский муниципальный район', 'Никольский муниципальный округ'),
    ('Никольский муниципальный округ', 'Никольский муниципальный район'),
    ('Спасский муниципальный район', 'Спасский'),
    ('Спасский муниципальный округ', 'Спасский'),
    ('Спасский муниципальный район', 'Спасский муниципальный округ'),
    ('Спасский муниципальный округ', 'Спасский муниципальный район'),
    ('Дмитровский муниципальный район', 'Дмитровский'),
    ('Дмитровский муниципальный округ', 'Дмитровский'),
    ('Дмитровский муниципальный район', 'Дмитровский муниципальный округ'),
    ('Беловский муниципальный округ', 'Беловский'),
    ('Беловский муниципальный округ', 'Беловский муниципальный район'),
    ('Кемеровский муниципальный округ', 'Кемеровский'),
    ('Новокузнецкий муниципальный округ', 'Новокузнецкий'),
    ('Прокопьевский муниципальный округ', 'Прокопьевский'),
    ('Сосновоборский муниципальный район', 'Сосновоборский'),
    ('Корсаковский муниципальный район', 'Корсаковский'),
    ('Чебаркульский муниципальный район', 'Чебаркульский'),
    ('Камышловский муниципальный район', 'Камышловский'),
    ('Чусовской муниципальный округ', 'Чусовской муниципальный округ'),
    ('Ильинский муниципальный район', 'Ильинский муниципальный округ'),
    ('Партизанский муниципальный район', 'Партизанский'),
    ('Партизанский муниципальный район', 'Партизанский муниципальный округ'),
    ('Пермский муниципальный район', 'Пермский'),
    ('Добрянский муниципальный район', 'Добрянский муниципальный округ'),
    ('Ленинск-Кузнецкий муниципальный округ', 'Ленинск-Кузнецкий муниципальный округ'),
}

pairs_clean = pairs[~pairs.apply(lambda r: (r['df_name'], r['gdf_name']) in blacklist, axis=1)].copy()
print(f"После чёрного списка: {len(pairs_clean)}")

# Приоритет, если у одного df_name несколько вариантов
def priority(row):
    d, g = row['df_name'], row['gdf_name']
    if 'район' in d and 'округ' in g:
        return 1
    if 'район' in d and 'округ' not in g and 'район' not in g:
        return 2
    if d.startswith('городской округ') and not g.startswith('городской округ'):
        return 3
    if 'округ' in d and 'район' in g:
        return 4
    return 5

pairs_clean['priority'] = pairs_clean.apply(priority, axis=1)
pairs_clean = pairs_clean.sort_values(['df_name', 'priority'])
pairs_final = pairs_clean.drop_duplicates(subset='df_name', keep='first')


pairs_final[['df_name', 'gdf_name']].to_csv(
    'replace_map_clean.csv', sep=';', index=False, encoding='utf-8'
)

print(f"\nИтог: {len(pairs_final)} уникальных пар")
print("Файл: replace_map_clean.csv")

df: 992 названий
gdf: 2566 названий


KeyboardInterrupt: 

Заменяем названия в gdf

In [15]:
rmap = pd.read_csv('replace_map_clean.csv', sep=';', encoding='utf-8')
rename_dict = dict(zip(rmap['gdf_name'], rmap['df_name']))

gdf['name_for_merge'] = gdf['name'].map(rename_dict).fillna(gdf['name'])

In [37]:
import plotly.express as px
import plotly.graph_objects as go

In [17]:
rmap = pd.read_csv(
    'replace_map_clean.csv',
    sep=';',
    encoding='utf-8'
)

rename_dict = dict(
    zip(rmap['gdf_name'], rmap['df_name'])
)

gdf['name_for_merge'] = (
    gdf['name']
    .map(rename_dict)
    .fillna(gdf['name'])
)

# Кластеры только за 2024 год

cluster_map = (
    df[
        (df['year'] == 2024) &
        (df['cluster'].notna())
    ][['mo', 'cluster']]
    .drop_duplicates('mo')
    .copy()
)

cluster_map['cluster'] = cluster_map['cluster'].astype(int)

print("Кластеры 2024:")
print(cluster_map['cluster'].value_counts().sort_index())
gdf_base = gdf[
    ['name_for_merge', 'geometry']
].copy()
merged = gdf_base.merge(
    cluster_map,
    left_on='name_for_merge',
    right_on='mo',
    how='left'
)

print(f"Территорий после merge: {len(merged)}")
print(merged.columns.tolist())

Кластеры 2024:
cluster
0    210
1    287
2    216
3    171
4    108
Name: count, dtype: int64
Территорий после merge: 2674
['name_for_merge', 'geometry', 'mo', 'cluster']


In [32]:
merged = merged.reset_index(drop=True)
merged['id'] = merged.index.astype(str)

map_gdf = merged[
    ['id', 'name_for_merge', 'cluster', 'geometry']
].copy()

map_gdf = map_gdf.to_crs(epsg=6933)

map_gdf['geometry'] = map_gdf.geometry.simplify(
    5000,
    preserve_topology=True
)

map_gdf = map_gdf.to_crs(epsg=4326)
geojson_all = map_gdf.__geo_interface__

print(
    "Размер GeoJSON:",
    round(len(str(geojson_all)) / 1024 / 1024, 2),
    "МБ"
)
plot_df = map_gdf.copy()

plot_df['cluster_map'] = (
    plot_df['cluster']
    .fillna(-1)
    .astype(int)
)

fig = go.Figure()

fig.add_trace(
    go.Choroplethmap(
        geojson=geojson_all,
        locations=plot_df['id'],
        z=plot_df['cluster_map'],
        text=plot_df['name_for_merge'],

        colorscale=[
            [0.000, '#FFFFFF'],
            [0.142, '#FFFFFF'],

            [0.142, '#E53935'],
            [0.285, '#E53935'],

            [0.285, '#FDD835'],
            [0.428, '#FDD835'],

            [0.428, '#43A047'],
            [0.571, '#43A047'],

            [0.571, '#1E88E5'],
            [0.714, '#1E88E5'],

            [0.714, '#FB8C00'],
            [0.857, '#FB8C00'],

            [0.857, '#8E44AD'],
            [1.000, '#8E44AD'],
        ],

        zmin=-1,
        zmax=5,

        showscale=False,

        marker_opacity=0.9,
        marker_line_width=0.5,
        marker_line_color='black',

        hovertemplate=(
            '<b>%{text}</b><br>'
            'Кластер: %{z}'
            '<extra></extra>'
        )
    )
)

fig.update_layout(
    map=dict(
    style='white-bg',
    center={
        'lat': 40,
        'lon': 165
    },
    zoom=1.8
),
    height=900,
    width=1400,
    margin=dict(l=0, r=0, t=60, b=0),
    title='Кластеры МО, 2024 год'
)

fig.show(renderer='browser')

Размер GeoJSON: 4.55 МБ


In [34]:
fig.write_html("map_clusters_2024.html")